In [ ]:
# Install required libraries

!pip install -q pandas numpy scikit-learn datasets huggingface_hub

In [ ]:
# Import required libraries

import os
import re
import json
import random
import pandas as pd

from google.colab import userdata
from sklearn.model_selection import train_test_split
from huggingface_hub import login, HfApi
from datasets import load_dataset, Features, Value

In [ ]:
# Login to Hugging Face

hf_token = userdata.get("HF_TOKEN")

login(token=hf_token)

repo_id = "BSVGK/Text_to_KG_Construction_Dataset"

api = HfApi()

print("Hugging Face login completed")
print("Dataset repository:", repo_id)

In [ ]:
# Define raw CSV columns as string

features = Features({
    "record_id": Value("string"),
    "source": Value("string"),
    "ocid": Value("string"),
    "title": Value("string"),
    "description": Value("string"),
    "buyer_name": Value("string"),
    "supplier_name": Value("string"),
    "contract_value": Value("string"),
    "award_date": Value("string"),
    "cpv_code": Value("string"),
    "cpv_description": Value("string"),
    "location": Value("string"),
    "description_len": Value("string")
})

In [ ]:
# Load raw CSV dataset from Hugging Face

raw_dataset = load_dataset(
    "csv",
    data_files="hf://datasets/BSVGK/Text_to_KG_Construction_Dataset/construction_contracts_clean.csv",
    features=features,
    token=hf_token
)

raw_df = raw_dataset["train"].to_pandas()

print("Raw dataset shape:", raw_df.shape)
print(raw_df.columns)

raw_df.head()

In [ ]:
# Clean column names

raw_df.columns = (
    raw_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(raw_df.columns)

In [ ]:
# Select only required 7 columns

selected_columns = [
    "buyer_name",
    "supplier_name",
    "contract_value",
    "award_date",
    "cpv_code",
    "cpv_description",
    "location"
]

df = raw_df[selected_columns].copy()

print("Selected dataset shape:", df.shape)
df.head()

In [ ]:
# Set clear data types

df["buyer_name"] = df["buyer_name"].astype("string")
df["supplier_name"] = df["supplier_name"].astype("string")
df["contract_value"] = pd.to_numeric(df["contract_value"], errors="coerce")
df["award_date"] = pd.to_datetime(df["award_date"], errors="coerce", utc=True).dt.date
df["cpv_code"] = df["cpv_code"].astype("string")
df["cpv_description"] = df["cpv_description"].astype("string")
df["location"] = df["location"].astype("string")

print(df.dtypes)
df.head()

In [ ]:
# Create output directory
from google.colab import drive
drive.mount('/content/drive')


output_dir = "/content/drive/MyDrive/Depixen/PROJECT_2_TEXT_TO_KG/construction_text_to_kg_output"

os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Create null summary before cleaning

null_summary_before = pd.DataFrame({
    "column_name": df.columns,
    "data_type": [str(df[col].dtype) for col in df.columns],
    "total_rows": [len(df)] * len(df.columns),
    "null_values": [df[col].isna().sum() for col in df.columns],
    "non_null_values": [df[col].notna().sum() for col in df.columns],
    "null_percentage": [round((df[col].isna().sum() / len(df)) * 100, 2) for col in df.columns]
})

null_summary_before.to_csv(
    f"{output_dir}/null_summary_before_cleaning.csv",
    index=False
)

null_summary_before

In [ ]:
# Clean text values

def clean_text(value):
    value = str(value)
    value = re.sub(r"\s+", " ", value)
    value = value.strip()
    return value

text_columns = [
    "buyer_name",
    "supplier_name",
    "cpv_code",
    "cpv_description",
    "location"
]

for col in text_columns:
    df[col] = df[col].apply(clean_text)

In [ ]:
# Replace invalid values with missing values

invalid_values = [
    "",
    " ",
    "nan",
    "none",
    "null",
    "na",
    "n/a",
    "N/A",
    "NA",
    "None",
    "NULL",
    "<NA>",
    "NaN",
    "-"
]

df = df.replace(invalid_values, pd.NA)

In [ ]:
# Format contract value and remove .0 from whole numbers

def format_contract_value(value):
    if pd.isna(value):
        return pd.NA

    value = float(value)

    if value.is_integer():
        return str(int(value))

    return str(round(value, 2))

df["contract_value"] = df["contract_value"].apply(format_contract_value)

In [ ]:
# Create clean dataframe with complete data in all selected columns

clean_df = df.dropna(subset=selected_columns).copy()

for col in selected_columns:
    clean_df[col] = clean_df[col].astype(str).str.strip()

clean_df = clean_df[
    (clean_df[selected_columns] != "").all(axis=1)
].copy()

clean_df = clean_df.reset_index(drop=True)

print("Original selected rows:", len(df))
print("Clean rows:", len(clean_df))
print("Removed rows:", len(df) - len(clean_df))

clean_df.head()

In [ ]:
# Final missing and empty value check

missing_check = clean_df[selected_columns].isna().sum()
empty_check = (clean_df[selected_columns].astype(str).apply(lambda x: x.str.strip()) == "").sum()

print("Missing values:")
print(missing_check)

print("\nEmpty values:")
print(empty_check)

In [ ]:
# Create null summary after cleaning

null_summary_after = pd.DataFrame({
    "column_name": clean_df.columns,
    "data_type": [str(clean_df[col].dtype) for col in clean_df.columns],
    "total_rows": [len(clean_df)] * len(clean_df.columns),
    "null_values": [clean_df[col].isna().sum() for col in clean_df.columns],
    "non_null_values": [clean_df[col].notna().sum() for col in clean_df.columns],
    "null_percentage": [round((clean_df[col].isna().sum() / len(clean_df)) * 100, 2) for col in clean_df.columns]
})

null_summary_after.to_csv(
    f"{output_dir}/null_summary_after_cleaning.csv",
    index=False
)

null_summary_after

In [ ]:
# Create unique contract ID for every clean row

clean_df["contract_id"] = [
    f"Contract_{str(i + 1).zfill(6)}" for i in range(len(clean_df))
]

clean_df.head()

In [ ]:
# Define ontology schema

ontology = {
    "classes": {
        "Contract": "A public construction contract record.",
        "Buyer": "The organisation purchasing or awarding the contract.",
        "Supplier": "The organisation awarded the contract.",
        "CPVCategory": "The procurement classification category.",
        "Location": "The geographical location linked to the contract."
    },
    "relationships": {
        "hasBuyer": "Links a contract to the buying organisation.",
        "hasSupplier": "Links a contract to the awarded supplier.",
        "hasContractValue": "Stores the monetary value of the contract.",
        "hasAwardDate": "Stores the contract award date.",
        "hasCPVCode": "Stores the CPV classification code.",
        "hasCPVDescription": "Stores the CPV category description.",
        "hasLocation": "Stores the contract location."
    }
}

with open(f"{output_dir}/ontology_schema.json", "w", encoding="utf-8") as f:
    json.dump(ontology, f, indent=4, ensure_ascii=False)

print("Ontology schema saved")

In [ ]:
# Create 10 natural language input templates
templates = [
    "Contract {contract_id} records that {buyer_name} awarded {supplier_name} a contract worth £{contract_value} on {award_date} for {cpv_description} in {location}, with CPV code {cpv_code}.",

    "For contract {contract_id}, {supplier_name} received a £{contract_value} contract from {buyer_name} on {award_date} in {location}, related to {cpv_description} and classified under CPV code {cpv_code}.",

    "Contract {contract_id} shows a £{contract_value} award from {buyer_name} to {supplier_name} on {award_date} for {cpv_description} in {location}, using CPV code {cpv_code}.",

    "{contract_id} refers to a contract secured by {supplier_name} from {buyer_name} for {cpv_description} in {location}. The award date was {award_date}, the value was £{contract_value}, and the CPV code was {cpv_code}.",

    "The contract identified as {contract_id} was awarded by {buyer_name} to {supplier_name} on {award_date}. It was valued at £{contract_value}, located in {location}, and linked to {cpv_description} under CPV code {cpv_code}.",

    "In {location}, contract {contract_id} was issued by {buyer_name} to {supplier_name} for {cpv_description}. The recorded value was £{contract_value}, the award date was {award_date}, and the CPV code was {cpv_code}.",

    "The record for {contract_id} shows that {supplier_name} was awarded work by {buyer_name} on {award_date}. The contract value is £{contract_value}, the location is {location}, and the CPV classification is {cpv_code} for {cpv_description}.",

    "{buyer_name} selected {supplier_name} for contract {contract_id} in {location}. The work is described as {cpv_description}, with CPV code {cpv_code}, awarded on {award_date}, and valued at £{contract_value}.",

    "A £{contract_value} contract identified as {contract_id} in {location} was awarded to {supplier_name} by {buyer_name}. The award date was {award_date}, and the contract is associated with CPV code {cpv_code} for {cpv_description}.",

    "For {cpv_description} in {location}, {buyer_name} awarded {supplier_name} contract {contract_id} on {award_date}. The contract value was £{contract_value}, and the CPV code was {cpv_code}."
]

In [ ]:
# Define instruction variants

instructions = [
    "Convert the given construction contract text into structured knowledge graph triples.",
    "Extract entities and relationships from the construction contract description and represent them as triples.",
    "Transform the provided contract information into a structured knowledge graph format.",
    "Identify key elements within the contract text and express them as subject-predicate-object triples.",
    "Generate a structured representation of the contract details in the form of knowledge graph triples.",
    "Analyse the contract description and map the relevant information into a triple-based format.",
    "Convert the natural language contract description into a structured set of relational triples.",
    "Extract and organise the contract information into a knowledge graph representation.",
    "Interpret the contract text and represent the contained information as structured triples.",
    "Produce a knowledge graph representation by extracting entities and their relationships from the contract description."
]

In [ ]:
# Set random seed

random.seed(42)

In [ ]:
# Generate natural language input text

def generate_input(row):
    template = random.choice(templates)

    return template.format(
        contract_id=row["contract_id"],
        buyer_name=row["buyer_name"],
        supplier_name=row["supplier_name"],
        contract_value=row["contract_value"],
        award_date=row["award_date"],
        cpv_code=row["cpv_code"],
        cpv_description=row["cpv_description"],
        location=row["location"]
    )

clean_df["input"] = clean_df.apply(generate_input, axis=1)

In [ ]:
# Generate ontology-based KG triples

def generate_output(row):
    triples = [
        f"({row['contract_id']}, rdf:type, Contract)",
        f"({row['contract_id']}, hasBuyer, {row['buyer_name']})",
        f"({row['contract_id']}, hasSupplier, {row['supplier_name']})",
        f"({row['contract_id']}, hasContractValue, {row['contract_value']})",
        f"({row['contract_id']}, hasAwardDate, {row['award_date']})",
        f"({row['contract_id']}, hasCPVCode, {row['cpv_code']})",
        f"({row['contract_id']}, hasCPVDescription, {row['cpv_description']})",
        f"({row['contract_id']}, hasLocation, {row['location']})"
    ]

    return "\n".join(triples)

clean_df["output"] = clean_df.apply(generate_output, axis=1)

In [ ]:
# Assign instruction to each row

clean_df["instruction"] = [
    random.choice(instructions) for _ in range(len(clean_df))
]

In [ ]:
# Create final training dataframe

training_df = clean_df[
    [
        "instruction",
        "input",
        "output"
    ]
].copy()

print("Training dataframe shape:", training_df.shape)

training_df.head()

In [ ]:
# Save full training dataset as CSV

training_df.to_csv(
    f"{output_dir}/full_training_dataset.csv",
    index=False
)

In [ ]:
# Check one generated sample

sample_index = 0

print("Instruction:")
print(training_df.iloc[sample_index]["instruction"])

print("\nInput:")
print(training_df.iloc[sample_index]["input"])

print("\nOutput:")
print(training_df.iloc[sample_index]["output"])

In [ ]:
# Split dataset into train validation and test

train_df, temp_df = train_test_split(
    training_df,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
# Save split datasets as CSV files

train_df.to_csv(
    f"{output_dir}/train.csv",
    index=False
)

valid_df.to_csv(
    f"{output_dir}/validation.csv",
    index=False
)

test_df.to_csv(
    f"{output_dir}/test.csv",
    index=False
)

print("CSV files saved")

In [ ]:
# Save split datasets as JSONL files

def save_jsonl(dataframe, file_path):
    with open(file_path, "w", encoding="utf-8") as f:
        for _, row in dataframe.iterrows():
            record = {
                "instruction": row["instruction"],
                "input": row["input"],
                "output": row["output"]
            }

            f.write(json.dumps(record, ensure_ascii=False) + "\n")

save_jsonl(train_df, f"{output_dir}/train.jsonl")
save_jsonl(valid_df, f"{output_dir}/validation.jsonl")
save_jsonl(test_df, f"{output_dir}/test.jsonl")

print("JSONL files saved")
print(os.listdir(output_dir))

In [ ]:
# Upload all processed files to Hugging Face

api.upload_folder(
    folder_path=output_dir,
    path_in_repo="processed",
    repo_id=repo_id,
    repo_type="dataset"
)

print("Processed files uploaded to Hugging Face successfully")

In [ ]:
# Load final processed dataset from Hugging Face

final_dataset = load_dataset(
    "json",
    data_files={
        "train": f"hf://datasets/{repo_id}/processed/train.jsonl",
        "validation": f"hf://datasets/{repo_id}/processed/validation.jsonl",
        "test": f"hf://datasets/{repo_id}/processed/test.jsonl"
    },
    token=hf_token
)

print(final_dataset)
print(final_dataset["train"][0])

In [ ]:
# Final summary

print("Dataset preparation completed")
print("Clean selected rows:", clean_df.shape)
print("Training dataset:", training_df.shape)
print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)
print("Repository:", repo_id)

In [ ]:
# Convert HF dataset to pandas for checking

train_df = final_dataset["train"].to_pandas()
valid_df = final_dataset["validation"].to_pandas()
test_df = final_dataset["test"].to_pandas()

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

In [ ]:
required_columns = ["instruction", "input", "output"]

print("Columns:", train_df.columns)

assert all(col in train_df.columns for col in required_columns), "Missing required columns"

In [ ]:
print("\nNull values (train):")
print(train_df.isna().sum())

print("\nEmpty values (train):")
print((train_df.astype(str).apply(lambda x: x.str.strip()) == "").sum())

In [ ]:
import re

def extract_id(text):
    match = re.search(r"(Contract_\d+)", str(text))
    return match.group(1) if match else None

train_df["input_id"] = train_df["input"].apply(extract_id)
train_df["output_id"] = train_df["output"].apply(extract_id)

mismatch = train_df[train_df["input_id"] != train_df["output_id"]]

print("\nContract ID mismatch count:", len(mismatch))

In [ ]:
def check_triples(output):
    lines = str(output).split("\n")
    return all(line.startswith("(") and line.endswith(")") for line in lines)

invalid_triples = train_df[~train_df["output"].apply(check_triples)]

print("Invalid triple rows:", len(invalid_triples))

In [ ]:
expected_relations = [
    "rdf:type",
    "hasBuyer",
    "hasSupplier",
    "hasContractValue",
    "hasAwardDate",
    "hasCPVCode",
    "hasCPVDescription",
    "hasLocation"
]

def check_relations(output):
    return all(rel in output for rel in expected_relations)

missing_rel = train_df[~train_df["output"].apply(check_relations)]

print("Rows missing relations:", len(missing_rel))

In [ ]:
for i in range(3):
    print("Instruction:\n", train_df.iloc[i]["instruction"])
    print("\nInput:\n", train_df.iloc[i]["input"])
    print("\nOutput:\n", train_df.iloc[i]["output"])
    print("\n------------------------\n")